**Problem description**: The team was tasked with developing a model to predict the busyness of a certain geographical region. The project uses a dataset containing the courier locations captured during food collection at restaurants during a time interval. The data scientist has produced a working Proof of Concept (PoC). Now, as an ML Engineer, you are tasked with productionizing this PoC.
This notebook contains the data scientist’s code to collect and create geo-location features to describe the busyness of regions (defined as h3 hexagons), and then train an ML model. The data scientist rushed to produce the PoC notebook, so the code is not well structured for a production application. As an ML engineer, your task is to:
1. Define a structured ML pipeline project.
2. Refactor the notebook into a project into productionized ML system using ML and software engineering best practices and appropriate tooling.

**Expected outcome**: A structured ML pipeline project in a Git repo that you will talk us through, explaining your design choices.
It should contain:
- Scripts for each step
- Training and prediction pipeline(s)
- Configurations file(s)
- Dependency management
- CI/CD

**Hints**:
We suggest containerization with Docker, using GCS for storage, Vertex AI or Airflow to execute the pipeline, and GitHub Actions for CI/CD. But if you feel more comfortable with other tools that is ok.


Consider creating files for each step, for example, `data_collection.py`, `feature_generation.py`, `training.py`, and `prediction.py`, in addition to pipeline and config files to connect and execute the pipeline. Some features might be poorly implemented or not be in use. Your focus as an ML Engineer is refactoring the notebook into a structure project, but you can highlight any implementation issues you identify.

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

# 1. Data collection

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/ani/Projects/JET_take_home_assignment/data/input/JET_take_home_assignment_raw_data.csv')
df.dropna(axis=0, inplace=True)

In [4]:
#unique couriers
len(df.courier_id.unique())

504

In [5]:
#unique restaurants
restaurants_ids = {}
list_restaurants_ids = []
for a,b in zip(df.restaurant_lat, df.restaurant_lon):
  id = "{}_{}".format(a,b)
  restaurants_ids[id] = {"lat": a, "lon":b}
for i,key in enumerate(restaurants_ids.keys()):
  restaurants_ids[key]['id'] = i

#labeling of restaurants
df['restaurant_id']=[restaurants_ids["{}_{}".format(a,b)]['id'] for a,b in zip(df.restaurant_lat, df.restaurant_lon)]
# number of unique restaurants
len(restaurants_ids)

268

# 2. Features
## 2.1 euclidean distance to restaurant


In [6]:
import collections.abc
# calc. eucl. distances to restaurants arrays
def calc_dist(p1x, p1y, p2x, p2y):
  p1 = (p2x - p1x)**2
  p2 = (p2y - p1y)**2
  dist = np.sqrt(p1 + p2)
  return dist.tolist() if isinstance(p1x, collections.abc.Sequence) else dist

df['dist_to_restaurant'] = calc_dist(df.courier_lat, df.courier_lon, df.restaurant_lat, df.restaurant_lon)

##2.2 avg. eucl. distance to restantaurants

In [7]:
# calc. avg. distance to restaurants
def avg_dist_to_restaurants(courier_lat,courier_lon):
  return np.mean([calc_dist(v['lat'], v['lon'], courier_lat, courier_lon) for v in restaurants_ids.values()])

df['avg_dist_to_restaurants'] = [avg_dist_to_restaurants(lat,lon) for lat,lon in zip(df.courier_lat, df.courier_lon)]

##2.3 Haversine distance to restaurant

In [8]:
from math import radians, cos, sin, asin, sqrt
import numpy as np

def calc_haversine_dist(lat1, lon1, lat2, lon2):

  R = 6372.8    #3959.87433  this is in miles.  For Earth radius in kilometers use 6372.8 km
  if isinstance(lat1, collections.abc.Sequence):
    dLat = np.array([radians(l2 - l1) for l2,l1 in zip(lat2, lat1)])
    dLon = np.array([radians(l2 - l1) for l2,l1 in zip(lon2, lon1)])
    lat1 = np.array([radians(l) for l in lat1])
    lat2 = np.array([radians(l) for l in lat2])
  else:
    dLat = radians(lat2 - lat1)
    dLon = radians(lon2 - lon1)
    lat1 = radians(lat1)
    lat2 = radians(lat2)

  a = np.sin(dLat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dLon/2)**2
  c = 2*np.arcsin(np.sqrt(a))
  dist = R*c
  return dist.tolist() if isinstance(lon1, collections.abc.Sequence) else dist

df['Hdist_to_restaurant'] = calc_haversine_dist(df.courier_lat.tolist(), df.courier_lon.tolist(), df.restaurant_lat.tolist(), df.restaurant_lon.tolist())

## 2.4 avg. Haversine distance to restantaurants

In [9]:
# calc. avg. distance to restaurants
def avg_Hdist_to_restaurants(courier_lat,courier_lon):
  return np.mean([calc_haversine_dist(v['lat'], v['lon'], courier_lat, courier_lon) for v in restaurants_ids.values()])

df['avg_Hdist_to_restaurants'] = [avg_Hdist_to_restaurants(lat,lon) for lat,lon in zip(df.courier_lat, df.courier_lon)]

##2.5 Five-Clusters embedding

In [10]:
#STEP 1 - define K & initiate data

def initiate_centroids(k, df):
    '''
    Select k data points as centroids
    k: number of centroids
    dset: pandas dataframe
    '''
    centroids = df.sample(k)
    return centroids

np.random.seed(1)
k=5
df_restaurants = pd.DataFrame([{"lat": v['lat'], "lon": v['lon']} for v in restaurants_ids.values()])
centroids = initiate_centroids(k, df_restaurants)

df_couriers = pd.DataFrame({})
df_couriers['lat'] = df['courier_lat']
df_couriers['lon'] = df['courier_lon']


# STEP 2 - define distance metric : Euclidean distance
def eucl_dist(p1x,p1y,p2x,p2y):
  return calc_dist(p1x, p1y, p2x, p2y)

# STEP 3 - Centroid assignment
def centroid_assignation(df, centroids):
  k = len(centroids)
  n = len(df)
  assignation = []
  assign_errors = []
  centroids_list = [c for i,c in centroids.iterrows()]
  for i,obs in df.iterrows():
    # Estimate error
    all_errors = [eucl_dist( centroid['lat'],
                            centroid['lon'],
                            obs['courier_lat'],
                            obs['courier_lon']) for centroid in centroids_list]

    # Get the nearest centroid and the error
    nearest_centroid =  np.where(all_errors==np.min(all_errors))[0].tolist()[0]
    nearest_centroid_error = np.min(all_errors)

    # Add values to corresponding lists
    assignation.append(nearest_centroid)
    assign_errors.append(nearest_centroid_error)
  df['Five_Clusters_embedding'] =assignation
  df['Five_Clusters_embedding_error'] =assign_errors
  return df

df = centroid_assignation(df,centroids)

##2.6 H3 clustering

In [13]:
# !pip install h3

In [14]:
df

,courier_id,order_number,courier_location_timestamp,courier_lat,courier_lon,order_created_timestamp,restaurant_lat,restaurant_lon,restaurant_id,dist_to_restaurant,avg_dist_to_restaurants,Hdist_to_restaurant,avg_Hdist_to_restaurants,Five_Clusters_embedding,Five_Clusters_embedding_error
0,a98737cbhoho5012hoho4b5bhoho867fhoho8475c658546d,281289453,2021-04-02T04:30:42.328Z,50.484520,-104.618876,2021-04-02T04:20:42Z,50.483696,-104.614350,0,0.004600,0.056686,0.333173,5.267000,0,0.028487
1,39a26fa0hohof428hoho47a4hohoa320hoho12e3d831c23a,280949566,2021-04-01T06:14:47.386Z,50.442573,-104.550463,2021-04-01T06:05:18Z,50.442422,-104.550487,1,0.000152,0.064685,0.016818,5.036710,3,0.006967
2,3813235ehoho7a42hoho4601hohob7eahoho799e8af5b535,281328578,2021-04-02T05:48:57.224Z,50.495920,-104.635605,2021-04-02T05:13:26Z,50.496595,-104.635606,2,0.000675,0.067024,0.075033,6.284221,0,0.008998
3,9f033953hohocd53hoho488ahohoaf51hohoc57943e499ed,281317998,2021-04-02T05:12:17.252Z,50.449445,-104.611521,2021-04-02T04:59:57Z,50.449504,-104.611074,3,0.000450,0.045752,0.032265,3.942031,4,0.035748
4,56f65bc8hohoba54hoho47dfhohoa09chohof7464b5d9848,281314132,2021-04-02T05:15:38.266Z,50.495254,-104.666383,2021-04-02T04:54:53Z,50.495160,-104.665733,4,0.000656,0.082413,0.047152,7.161970,0,0.021886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20572,2f21e0c7hoho79b8hoho4ecdhohoaacbhohoad8b3e6565c2,281356256,2021-04-02T05:49:45.288Z,50.463855,-104.618036,2021-04-02T05:48:14Z,50.475204,-104.617475,267,0.011363,0.048363,1.262957,4.254325,0,0.042334
20573,b3fe5a77hohofbb2hoho4c5fhohob9d5hoho1f87dd2ab9ed,281348386,2021-04-02T05:38:50.548Z,50.482136,-104.606574,2021-04-02T05:38:29Z,50.475204,-104.617475,267,0.012918,0.055359,1.090740,5.110064,0,0.040733
20574,30a6cb7fhoho1825hoho407ehoho97f0hoho9d374a6b2f20,281353044,2021-04-02T05:46:23.316Z,50.473982,-104.631716,2021-04-02T05:44:18Z,50.475204,-104.617475,267,0.014293,0.055344,1.017159,4.891502,0,0.026228
20575,30a6cb7fhoho1825hoho407ehoho97f0hoho9d374a6b2f20,281313038,2021-04-02T04:53:53.119Z,50.466229,-104.618022,2021-04-02T04:53:22Z,50.475204,-104.617475,267,0.008992,0.048937,0.999056,4.328311,0,0.040520


In [18]:
import h3

resolution=7
df['courier_location_timestamp']=  pd.to_datetime(df['courier_location_timestamp'], format='mixed')
df['order_created_timestamp'] = pd.to_datetime(df['order_created_timestamp'])
df['h3_index'] = [h3.geo_to_h3(lat,lon,resolution) for (lat,lon) in zip(df.courier_lat, df.courier_lon)]
df['date_day_number'] = [d for d in df.courier_location_timestamp.dt.day_of_year]
df['date_hour_number'] = [d for d in df.courier_location_timestamp.dt.hour]

## 2.7 Orders busyness

In [19]:
index_list = [(i,d,hr) for (i,d,hr) in zip(df.h3_index, df.date_day_number, df.date_hour_number)]

set_indexes = list(set(index_list))
dict_indexes = {label: index_list.count(label) for label in set_indexes}
df['orders_busyness_by_h3_hour'] = [dict_indexes[i] for i in index_list]

##2.8 number de restuarants per h3 index

In [20]:
restaurants_counts_per_h3_index = {a:len(b) for a,b in zip(df.groupby('h3_index')['restaurant_id'].unique().index, df.groupby('h3_index')['restaurant_id'].unique()) }
df['restaurants_per_index'] = [restaurants_counts_per_h3_index[h] for h in df.h3_index]

##2.9 Label encoding

In [21]:
from sklearn.preprocessing import LabelEncoder

def Encoder(df):
  columnsToEncode = list(df.select_dtypes(include=['category','object']))
  le = LabelEncoder()
  for feature in columnsToEncode:
      try:
          df[feature] = le.fit_transform(df[feature])
      except:
          print('Error encoding '+feature)
  return df

df['h3_index'] = df.h3_index.astype('category')

df= Encoder(df)

In [22]:
df.head()

,courier_id,order_number,courier_location_timestamp,courier_lat,courier_lon,order_created_timestamp,restaurant_lat,restaurant_lon,restaurant_id,dist_to_restaurant,avg_dist_to_restaurants,Hdist_to_restaurant,avg_Hdist_to_restaurants,Five_Clusters_embedding,Five_Clusters_embedding_error,h3_index,date_day_number,date_hour_number,orders_busyness_by_h3_hour,restaurants_per_index
0,346,281289453,2021-04-02 04:30:42.328000+00:00,50.484520,-104.618876,2021-04-02 04:20:42+00:00,50.483696,-104.614350,0,0.004600,0.056686,0.333173,5.267000,0,0.028487,28,92,4,422,51
1,116,280949566,2021-04-01 06:14:47.386000+00:00,50.442573,-104.550463,2021-04-01 06:05:18+00:00,50.442422,-104.550487,1,0.000152,0.064685,0.016818,5.036710,3,0.006967,17,91,6,701,75
2,110,281328578,2021-04-02 05:48:57.224000+00:00,50.495920,-104.635605,2021-04-02 05:13:26+00:00,50.496595,-104.635606,2,0.000675,0.067024,0.075033,6.284221,0,0.008998,6,92,5,941,59
3,328,281317998,2021-04-02 05:12:17.252000+00:00,50.449445,-104.611521,2021-04-02 04:59:57+00:00,50.449504,-104.611074,3,0.000450,0.045752,0.032265,3.942031,4,0.035748,22,92,5,1026,69
4,178,281314132,2021-04-02 05:15:38.266000+00:00,50.495254,-104.666383,2021-04-02 04:54:53+00:00,50.495160,-104.665733,4,0.000656,0.082413,0.047152,7.161970,0,0.021886,5,92,5,397,35


#3. data preparation, trainig & validation

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X = df[['dist_to_restaurant', 'Hdist_to_restaurant', 'avg_Hdist_to_restaurants', 'date_day_number', 'restaurant_id', 'Five_Clusters_embedding', 'h3_index','date_hour_number', 'restaurants_per_index']]
y = df[['orders_busyness_by_h3_hour']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)


regr = RandomForestRegressor(max_depth=4, random_state=0, n_jobs=-1)


In [24]:
regr.fit(X_train, y_train)
regr.score(X_test, y_test)

/Users/ani/Projects/JET_take_home_assignment/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


0.9122340438119385

In [25]:
params = {
    'max_depth': [4,5],
    'min_samples_leaf': [50,75],
    'n_estimators': [100,150]
}
from sklearn.model_selection import GridSearchCV
# Instantiate the grid search model
grid_search = GridSearchCV(estimator=regr,
                           param_grid=params,
                           cv = 3,
                           n_jobs=-1, verbose=1, scoring="r2")

In [26]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits


/Users/ani/Projects/JET_take_home_assignment/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/ani/Projects/JET_take_home_assignment/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/ani/Projects/JET_take_home_assignment/.venv/lib/python3.11/site-packages/sklearn/base.py:1336: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/Users/ani/Projects/JET_take_home_assignment/.venv/lib/python3.11/site-packages/sklea

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...andom_state=0)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [4, 5], 'min_samples_leaf': [50, 75], 'n_estimators': [100, 150]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is 

In [27]:
grid_search.best_score_

np.float64(0.9599586176921253)

In [28]:
rf_best = grid_search.best_estimator_
rf_best

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",50
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [29]:
rf_best.score(X_test, y_test)

0.9621476918712052